In [ ]:
%%writefile vector_add4.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(int *A, int *B, int *C, int n) {
    // Calculate global thread ID
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check to ensure we don't access memory outside the vector
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n = 1024;
    size_t size = n * sizeof(int);

    // Host memory allocation
    int h_A[1024], h_B[1024], h_C[1024];

    // Initialize vectors on host
    for (int i = 0; i < n; i++) {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    // Device memory allocation
    int *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data from Host to Device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Define execution configuration
    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    // Launch the GPU Kernel
    vectorAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, n);

    // Copy result back from Device to Host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Verify first 10 results
    printf("First 10 Results:\n");
    for (int i = 0; i < 10; i++) {
        printf("%d + %d = %d\n", h_A[i], h_B[i], h_C[i]);
    }

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector_add.cu


In [2]:
# Compile the .cu file into an executable
!nvcc vector_add.cu -o vector_add

# Run the executable
!./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
First 10 Results:
0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27
